# Gen Relevant Topics

* Given a bunch of knowledge base, and user specified topics --> for each topic, find the most relevant text in each documents
  * rank the retrieved text of needed

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from dotenv import load_dotenv
from chonkie import SemanticChunker

load_dotenv("../.env")  # loads OPENAI_API_KEY from .env file


True

In [5]:
selected_docs = ['0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt',
                '0a7f058fad69b8be410fed4859f3be4b4fc5d55cf631e4f0f3409985f78d26bb.txt',
                '0a0bcc5de9b3f7f28b94f8a23b2d620eb4372a61fbcb69604993123bc59fc2a3.txt',
                'fffe1b498f9816a054db6841ad9bf85cdc18bb36cba5b62f6275f1b386d50be3.txt',
]
wix_docs_path = '../datasets/wix_docs'

for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    token_count = len(text) // 4
    print(f"{doc}: ~{token_count} tokens (estimated at 4 chars/token)")


0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: ~1262 tokens (estimated at 4 chars/token)
0a7f058fad69b8be410fed4859f3be4b4fc5d55cf631e4f0f3409985f78d26bb.txt: ~1067 tokens (estimated at 4 chars/token)
0a0bcc5de9b3f7f28b94f8a23b2d620eb4372a61fbcb69604993123bc59fc2a3.txt: ~80 tokens (estimated at 4 chars/token)
fffe1b498f9816a054db6841ad9bf85cdc18bb36cba5b62f6275f1b386d50be3.txt: ~348 tokens (estimated at 4 chars/token)


## Chunking

* Langchain basic text spliters: https://docs.langchain.com/oss/python/integrations/splitters
* Chonkie chunkers: https://docs.chonkie.ai/oss/chunkers/overview

In [5]:
def _print_wrapped(text, words_per_line=50):
    if not text:
        return
    words = str(text).split()
    for i in range(0, len(words), words_per_line):
        print(' '.join(words[i:i+words_per_line]))

#### Semantic Chunking

* `pip install "chonkie[semantic]"` available up to python 3.13, doesn't support newer python
* `pip install sentence-transformers`
* `pip install "chonkie[neural]"`

* Oberservations 🍀
  * Chonkie's AutoEmbeddings, looks like allows different embedding models but for openAI, somehow it will load from HuggingFace and gets error, so finally I had to use OpenAIEmbeddings directly from chonkie.
  * `SemanticChunker` uses the embedding model's tokenizer internally for token counting, and chonkie's OpenAIEmbeddings tokenizer wrapper isn't compatible with it. Passing a sentence-transformer model name string (like "all-MiniLM-L6-v2") works correctly and is what chonkie recommends for semantic chunking. 

In [10]:
chunker = SemanticChunker(
    embedding_model='all-MiniLM-L6-v2',          # sentence-transformer model, decide where to split
    threshold=0.8,                               # Similarity threshold (0-1)
    chunk_size=1200,                             # Maximum tokens per chunk
    similarity_window=3,                         # Window for similarity calculation
    skip_window=0                                # Skip-and-merge window (0=disabled)
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9207.05it/s]
/Users/hanhanwu/Documents/Github/Yokan/.venv13/lib/python3.13/site-packages/chonkie/embeddings/sentence_transformer.py:62: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self._dimension = self.model.get_sentence_embedding_dimension()


In [ ]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: 16 chunks
Facebook Ads: Creating a New Campaign Create targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook
Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience.
----CHUNK SEPARATOR----
Wix's AI also creates lookalike audiences to help get more eyes on your business.Before you begin:Make sure you've already done the following: Connected your Facebook
Page Upgraded to a Premium Plan Published your site Connected a domainLearn how get your site ready to launch a Facebook Ads campaign.Step 1 |
Choose a campaign goalOnce you've connected your Facebook page to Wix, the first step of creating a campaign is choosing a campaign goal. Depending on
your business you can focus on getting more store orders, bookings to your site, or to increase views and traffic to a page.Important:Once you choose
a campa

#### Sentence Chunking

* `pip install chonkie` (included in base install)
* Splits text into chunks based on sentence boundaries, respecting token limits

In [ ]:
from chonkie import SentenceChunker

# Basic initialization with default parameters
sentence_chunker = SentenceChunker(
    tokenizer="character",     # Default tokenizer (or use "gpt2", etc.)
    chunk_size=2048,           # Maximum tokens per chunk
    chunk_overlap=128,         # Overlap between chunks
    min_sentences_per_chunk=1  # Minimum sentences in each chunk
)

In [ ]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = sentence_chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

#### Neural Chunking

* `pip install "chonkie[neural]"`
* Uses a fine-tuned ML model to detect semantically meaningful chunk boundaries

In [ ]:
from chonkie import NeuralChunker

#CPU: 4.9s
neural_chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",  # Default model
    device_map="cpu",                        # Device to run the model on ('cpu', 'cuda', etc.)
    min_characters_per_chunk=10,             # Minimum characters for a chunk
)

/Users/hanhanwu/Documents/Github/Yokan/.venv13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 6997.27it/s]


In [ ]:
import torch

# Use MPS (Metal Performance Shaders) for Apple Silicon GPU
#GPU: 1.1s
device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

neural_chunker_gpu = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",
    device_map=device,
    min_characters_per_chunk=10,
)


Using device: mps


Loading weights: 100%|██████████| 138/138 [00:00<00:00, 1131.19it/s]


In [7]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = neural_chunker.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: 13 chunks
Facebook Ads: Creating a New Campaign Create targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook
Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience. Wix's AI also creates lookalike
audiences to help get more eyes on your business.
----CHUNK SEPARATOR----
Before you begin:Make sure you've already done the following: Connected your Facebook Page Upgraded to a Premium Plan Published your site Connected a domainLearn how
get your site ready to launch a Facebook Ads campaign.
----CHUNK SEPARATOR----
Step 1 | Choose a campaign goalOnce you've connected your Facebook page to Wix, the first step of creating a campaign is choosing a campaign
goal. Depending on your business you can focus on getting more store orders, bookings to your site, or to increase views and traffic to a
page.Importa

In [9]:
for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, 'r', encoding='utf-8') as f:
        text = f.read()
    chunks = neural_chunker_gpu.chunk(text)
    print(f"{doc}: {len(chunks)} chunks")
    for chunk in chunks:
        _print_wrapped(chunk, 25)
        print("----CHUNK SEPARATOR----")
    print("--------------------------DOC SEPARATOR------------------------")

0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt: 13 chunks
Facebook Ads: Creating a New Campaign Create targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook
Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience. Wix's AI also creates lookalike
audiences to help get more eyes on your business.
----CHUNK SEPARATOR----
Before you begin:Make sure you've already done the following: Connected your Facebook Page Upgraded to a Premium Plan Published your site Connected a domainLearn how
get your site ready to launch a Facebook Ads campaign.
----CHUNK SEPARATOR----
Step 1 | Choose a campaign goalOnce you've connected your Facebook page to Wix, the first step of creating a campaign is choosing a campaign
goal. Depending on your business you can focus on getting more store orders, bookings to your site, or to increase views and traffic to a
page.Importa

## Save Embedded Chunks

* Chose Neural Chunker for chunking here

In [1]:
%pip install supabase openai --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### Supabase Table Setup

Run the following SQL once in your Supabase SQL editor to create the `embeddings` table:

```sql
CREATE EXTENSION IF NOT EXISTS vector;

CREATE TABLE IF NOT EXISTS embeddings (
    id         bigserial PRIMARY KEY,
    doc_name   text        NOT NULL,
    chunk_index integer    NOT NULL,
    content    text        NOT NULL,
    embedding  vector(1536),
    created_at timestamptz DEFAULT now()
);

-- Optional: IVFFlat index for fast cosine similarity search
CREATE INDEX ON embeddings USING ivfflat (embedding vector_cosine_ops) WITH (lists = 100);
```

In [7]:
import os
import torch
from dotenv import load_dotenv
from openai import OpenAI
from supabase import create_client, Client
from chonkie import NeuralChunker

load_dotenv("../.env")

openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
supabase_client: Client = create_client(
    os.environ["SUPABASE_URL"],
    os.environ["SUPABASE_SERVICE_ROLE_KEY"],
)

device = "mps" if torch.backends.mps.is_available() else "cpu"
neural_chunker = NeuralChunker(
    model="mirth/chonky_modernbert_base_1",
    device_map=device,
    min_characters_per_chunk=10,
)

print(f"Clients initialized. NeuralChunker running on: {device}")

/Users/hanhanwu/Documents/Github/Yokan/.venv13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 138/138 [00:00<00:00, 552.86it/s]

Clients initialized. NeuralChunker running on: mps


In [11]:
EMBEDDING_MODEL = "text-embedding-3-small"
BATCH_SIZE = 100  # OpenAI allows up to 2048; Supabase insert batch size


def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a list of texts using OpenAI text-embedding-3-small."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]


records = []

for doc in selected_docs:
    filepath = os.path.join(wix_docs_path, doc)
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = neural_chunker.chunk(text)
    chunk_texts = [chunk.text for chunk in chunks]

    print(f"[{doc}] Chunked into {len(chunks)} chunks. Embedding...")
    embeddings = embed_texts(chunk_texts)

    for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
        records.append(
            {
                "doc_name": doc,
                "chunk_index": i,
                "content": chunk.text,
                "embedding": embedding,
            }
        )

print(f"\nTotal records prepared: {len(records)}")

# Upsert into Supabase in batches
for i in range(0, len(records), BATCH_SIZE):
    batch = records[i : i + BATCH_SIZE]
    supabase_client.table("embeddings").insert(batch).execute()
    print(f"  Inserted batch {i // BATCH_SIZE + 1} ({len(batch)} records)")

print("\nDone! All embeddings saved to Supabase table 'embeddings'.")

[0a6fd82793a145494170ae1896c53841d75a4eee743bfa818e3b0288e79ec5a3.txt] Chunked into 13 chunks. Embedding...
[0a7f058fad69b8be410fed4859f3be4b4fc5d55cf631e4f0f3409985f78d26bb.txt] Chunked into 14 chunks. Embedding...
[0a0bcc5de9b3f7f28b94f8a23b2d620eb4372a61fbcb69604993123bc59fc2a3.txt] Chunked into 2 chunks. Embedding...
[fffe1b498f9816a054db6841ad9bf85cdc18bb36cba5b62f6275f1b386d50be3.txt] Chunked into 6 chunks. Embedding...

Total records prepared: 35
  Inserted batch 1 (35 records)

Done! All embeddings saved to Supabase table 'embeddings'.


## Topics Matching

* **Load Chunks & Embeddings:** Fetch all chunk texts and pre-computed embeddings from Supabase in a single batch query.
* **Build BM25 Index:** Tokenize chunk texts and build a BM25 index locally using the `rank-bm25` library for fast keyword matching.
* **Embed Search Topics:** Generate embeddings for the target search topics in a single batch call via OpenAI's `text-embedding-3-small` model.
* **Hybrid Scoring:**
  - *Semantic Score:* Multiplies topic and chunk embeddings in NumPy to get vectorized cosine similarity.
  - *Keyword Score:* Queries the BM25 index with the tokenized topic text.
* **Rank Fusion (RRF):** Combine the two separate rankings using Reciprocal Rank Fusion ($k=60$) to obtain higher ranking quality than either method alone.
* **Fine-Grained Passage Extraction:** Slide a $3$-sentence window over each top chunk to score and extract the most precise, context-rich matching text.

### Why Semantic and Keyword Scores Have Equal Weight
- Reciprocal Rank Fusion operates on relative ordering rather than raw scores, naturally balancing different retrieval modalities by penalizing documents strictly based on their rank position without requiring manual weight tuning or normalization.
- **Rank-Based, Not Score-Based Fusion:** RRF combines rankings (ordered lists of items) instead of raw similarity/match scores. This design choice bypasses the need to normalize and calibrate dissimilar metrics (like bounds-free BM25 scores versus [-1, 1] cosine scores).
- **Sum of Reciprocal Positions:** In RRF, each document's rank list contribution is scored using the mathematical formula:
  $$RRF(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$
  Where $M = \{\text{Semantic List}, \text{Keyword List}\}$, and $r_m(d)$ is the 0-based rank of page $d$ in list $m$. 
- **Equal Modality Contribution:** Both lists are processed through the same loops and use the exact same position penalty term ($k=60$). As a result, being ranked 1st in Semantic yields the same reciprocal rank score ($1/61 \approx 0.01639$) as being ranked 1st in the Keyword list. This provides a robust, scale-invariant baseline without requiring arbitrary weight tuning.
- Weight tuning would only help once you have eval metrics to optimize against.

In [4]:
%pip install rank-bm25 pandas --quiet


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
topics = ['Facebook ads', 'targeted campaigns', 
          'audience segmentation', 'ad performance', 'conversion tracking']

In [8]:
import numpy as np
import pandas as pd
from collections import defaultdict

# Fetch all chunks + embeddings from Supabase in one call
response = (
    supabase_client.table("embeddings")
    .select("doc_name, chunk_index, content, embedding")
    .limit(10000)
    .execute()
)
chunks = response.data
print(f"Fetched {len(chunks)} chunks from Supabase")

Fetched 35 chunks from Supabase


In [10]:
import json
from rank_bm25 import BM25Okapi

EMBEDDING_MODEL = "text-embedding-3-small"

def embed_texts(texts: list[str]) -> list[list[float]]:
    """Embed a list of texts using OpenAI text-embedding-3-small."""
    response = openai_client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [item.embedding for item in response.data]

def parse_embedding(emb) -> list[float]:
    """Parse embedding — Supabase may return vectors as JSON strings."""
    if isinstance(emb, str):
        return json.loads(emb)
    return emb

# Build BM25 index on chunk texts
corpus_texts = [c["content"] for c in chunks]
tokenized_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)

# Stack chunk embeddings into a numpy matrix (N, 1536)
chunk_embeddings = np.array([parse_embedding(c["embedding"]) for c in chunks], dtype=float)

# Embed all topics in ONE batch call
topic_embeddings = np.array(embed_texts(topics))  # (5, 1536)

print(f"BM25 index built over {len(chunks)} chunks")
print(f"chunk_embeddings shape: {chunk_embeddings.shape}")
print(f"topic_embeddings shape: {topic_embeddings.shape}")

BM25 index built over 35 chunks
chunk_embeddings shape: (35, 1536)
topic_embeddings shape: (5, 1536)


In [11]:
import re

# ── helpers ──────────────────────────────────────────────────────────────────

def cosine_sim_batch(topic_vecs: np.ndarray, chunk_vecs: np.ndarray) -> np.ndarray:
    """Vectorized cosine similarity: (T, D) x (N, D) -> (T, N)."""
    topic_norms = np.linalg.norm(topic_vecs, axis=1, keepdims=True)  # (T,1)
    chunk_norms = np.linalg.norm(chunk_vecs, axis=1, keepdims=True)  # (N,1)
    dot = topic_vecs @ chunk_vecs.T  # (T, N)
    return dot / np.maximum(topic_norms * chunk_norms.T, 1e-10)


def rrf_fuse(rankings: list[list[int]], k: int = 60) -> dict[int, float]:
    """Reciprocal Rank Fusion across multiple ranked lists."""
    scores: dict[int, float] = defaultdict(float)
    for ranking in rankings:
        for rank, idx in enumerate(ranking):
            scores[idx] += 1.0 / (k + rank + 1)
    return scores


SENTENCE_ABBREVIATIONS = {"e.g", "i.e", "mr", "mrs", "ms", "dr", "prof", "sr", "jr", "vs", "etc"}


def _previous_token(text: str, index: int) -> str:
    i = index - 1
    while i >= 0 and text[i].isspace():
        i -= 1
    end = i + 1
    while i >= 0 and (text[i].isalpha() or text[i] == "."):
        i -= 1
    return text[i + 1 : end].lower()


def _next_non_space_index(text: str, index: int) -> int:
    i = index
    while i < len(text) and text[i].isspace():
        i += 1
    return i


def _is_sentence_end(text: str, index: int) -> bool:
    ch = text[index]
    if ch == "\n":
        return True
    if ch not in ".!?":
        return False

    token = _previous_token(text, index)
    if token in SENTENCE_ABBREVIATIONS:
        return False
    if index > 0 and index + 1 < len(text) and text[index - 1].isdigit() and text[index + 1].isdigit():
        return False

    after_punctuation = index + 1
    while after_punctuation < len(text) and text[after_punctuation] in '\"\')]' :
        after_punctuation += 1

    next_idx = _next_non_space_index(text, after_punctuation)
    return next_idx >= len(text) or text[next_idx] == "\n" or text[next_idx].isupper() or text[next_idx].isdigit()


def split_sentences(text: str) -> list[str]:
    """Split text into sentence-like spans, including glued boundaries like Ad.Step."""
    sentences: list[str] = []
    start = 0
    for i in range(len(text)):
        if not _is_sentence_end(text, i):
            continue
        end = i + 1
        while end < len(text) and text[end] in '\"\')]' :
            end += 1
        sentence = text[start:end].strip()
        if len(sentence) > 10:
            sentences.append(sentence)
        start = end
        while start < len(text) and text[start].isspace():
            start += 1

    tail = text[start:].strip()
    if len(tail) > 10:
        sentences.append(tail)
    return sentences


def score_window(window_text: str, topic_words: set[str], topic: str) -> float:
    """Score a passage window by term overlap + exact phrase bonus."""
    lower = window_text.lower()
    tokens = set(lower.split())
    overlap = len(topic_words & tokens) / max(len(topic_words), 1)
    phrase_bonus = 0.3 if topic.lower() in lower else 0.0
    return overlap + phrase_bonus


def extract_best_passage(chunk_text: str, topic: str, window: int = 3) -> str:
    """
    Split chunk into sentences, slide a K-sentence window,
    return the highest-scoring window as the matched passage.
    Falls back to first `window` sentences if chunk is very short.
    """
    sentences = split_sentences(chunk_text)
    if len(sentences) <= window:
        return " ".join(sentences)

    topic_words = set(topic.lower().split())
    best_score = -1.0
    best_passage = ""

    for i in range(len(sentences) - window + 1):
        window_sents = sentences[i : i + window]
        window_text = " ".join(window_sents)
        s = score_window(window_text, topic_words, topic)
        if s > best_score:
            best_score = s
            best_passage = window_text

    return best_passage


# ── hybrid search ─────────────────────────────────────────────────────────────

# All cosine similarities at once: (5, N)
sem_scores_all = cosine_sim_batch(topic_embeddings, chunk_embeddings)

results: dict[str, list[dict]] = {}

for t_idx, topic in enumerate(topics):
    sem_scores = sem_scores_all[t_idx]  # (N,)
    bm25_scores = bm25.get_scores(topic.lower().split())  # (N,)

    sem_ranking = np.argsort(-sem_scores).tolist()
    kw_ranking = np.argsort(-bm25_scores).tolist()

    fused = rrf_fuse([sem_ranking, kw_ranking])
    top3 = sorted(fused, key=lambda i: -fused[i])[:3]

    results[topic] = [
        {
            "rank": r + 1,
            "rrf_score": round(fused[i], 5),
            "semantic_score": round(float(sem_scores[i]), 4),
            "bm25_score": round(float(bm25_scores[i]), 4),
            "doc_name": chunks[i]["doc_name"][:20] + "...",
            "chunk_index": chunks[i]["chunk_index"],
            "matched_text": extract_best_passage(chunks[i]["content"], topic),
        }
        for r, i in enumerate(top3)
    ]

print("Search complete.")

Search complete.


In [12]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", 20)

for topic, rows in results.items():
    print(f"\n{'━' * 72}")
    print(f"  Topic: {topic}")
    print(f"{'━' * 72}")
    df = pd.DataFrame(rows)[
        ["rank", "rrf_score", "semantic_score", "bm25_score", "doc_name", "chunk_index", "matched_text"]
    ]
    display(df)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Topic: Facebook ads
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,rank,rrf_score,semantic_score,bm25_score,doc_name,chunk_index,matched_text
0,1,0.03252,0.6509,4.0769,0a6fd82793a145494170...,0,"Facebook Ads: Creating a New Campaign\nCreate targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience. Wix's AI also creates lookalike audiences to help get more eyes on your business."
1,2,0.03202,0.5323,4.4435,0a6fd82793a145494170...,1,Before you begin:Make sure you've already done the following:\nConnected your Facebook Page Upgraded to a Premium Plan Published your site Connected a domainLearn how get your site ready to launch a Facebook Ads campaign.
2,3,0.03102,0.4926,3.2453,0a6fd82793a145494170...,12,Review your payment details and click Submit Purchase.Your campaign now goes into review for Wix and Facebook approval. This process could take up to 72 hours. Learn more about the review process.Want some inspiration creating your campaign?Here are some great examples of Facebook Ads with Wix campaigns.



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Topic: targeted campaigns
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,rank,rrf_score,semantic_score,bm25_score,doc_name,chunk_index,matched_text
0,1,0.03279,0.4718,4.9313,0a6fd82793a145494170...,0,"Facebook Ads: Creating a New Campaign\nCreate targeted campaigns for Facebook and Instagram and get your business in front of thousands of people. Your Facebook Ads campaign is entirely in your control: design your ad, choose a monthly budget and set a specific target audience. Wix's AI also creates lookalike audiences to help get more eyes on your business."
1,2,0.02976,0.3161,2.5915,0a7f058fad69b8be410f...,7,Traffic by locationSee where in the world you are getting the most traffic. You can filter the information by country map or city map.Use this report to:\nOptimize your marketing efforts or paid campaigns to the locations where you are getting more traffic. Prioritize locations where you want to increase your marketing presence.
2,3,0.02946,0.2858,2.8860,0a7f058fad69b8be410f...,9,"Traffic by deviceSee if people are viewing your site from desktop, tablet, or mobile.Use this report to:\nMake sure your site on mobile is up-to-date and functions for your mobile visitors. Create offers and content that are targeted to specific devices."



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Topic: audience segmentation
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,rank,rrf_score,semantic_score,bm25_score,doc_name,chunk_index,matched_text
0,1,0.03279,0.4782,2.5963,0a6fd82793a145494170...,8,Drag the Age range slider to set the age range you want to target for your audience. (Optional) Add up to 10 interests for your target audience. These are things your audience might search for online.
1,2,0.03226,0.4338,2.3241,0a6fd82793a145494170...,7,"Step 3 | Choose your ad audiencePick a target audience using parameters like gender, age range and location. Once you pick an audience, Wix's AI uses the information to create lookalike audiences to help expose your campaign to more people. Learn more about choosing the right ad audience."
2,3,0.03102,0.2766,0.0000,0a7f058fad69b8be410f...,9,"Traffic by deviceSee if people are viewing your site from desktop, tablet, or mobile.Use this report to:\nMake sure your site on mobile is up-to-date and functions for your mobile visitors. Create offers and content that are targeted to specific devices."



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Topic: ad performance
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,rank,rrf_score,semantic_score,bm25_score,doc_name,chunk_index,matched_text
0,1,0.03252,0.2053,2.5694,0a6fd82793a145494170...,4,"Step 2 | Customize your ad contentChoose an ad format for your campaign and update the ad's creative. Depending on the ad format you choose, you will see different options.Note:You can only choose an ad format for the 'more store orders' campaign goal (if you have an online store). For the 'promote a page' goal or 'more leads or bookings' goals, you only have the option of creating a classic ad."
1,2,0.03202,0.1695,2.6860,0a6fd82793a145494170...,5,To customize your ad content: \nSelect the ad format you want to use for your campaign. Learn more about what ad format to choose. For dynamic ads:
2,3,0.03175,0.1896,2.2262,0a6fd82793a145494170...,7,"Step 3 | Choose your ad audiencePick a target audience using parameters like gender, age range and location. Once you pick an audience, Wix's AI uses the information to create lookalike audiences to help expose your campaign to more people. Learn more about choosing the right ad audience."



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Topic: conversion tracking
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


,rank,rrf_score,semantic_score,bm25_score,doc_name,chunk_index,matched_text
0,1,0.03150,0.3182,0.0,0a7f058fad69b8be410f...,7,Traffic by locationSee where in the world you are getting the most traffic. You can filter the information by country map or city map.Use this report to:\nOptimize your marketing efforts or paid campaigns to the locations where you are getting more traffic. Prioritize locations where you want to increase your marketing presence.
1,2,0.03128,0.3607,0.0,0a7f058fad69b8be410f...,9,"Traffic by deviceSee if people are viewing your site from desktop, tablet, or mobile.Use this report to:\nMake sure your site on mobile is up-to-date and functions for your mobile visitors. Create offers and content that are targeted to specific devices."
2,3,0.03080,0.2915,0.0,0a7f058fad69b8be410f...,6,Traffic by entry pageSee which site page visitors land on first when entering your site. Tip: Make sure the most important call-to-actions are visible on the pages that visitors are landing on the most.
